# 🎓 Recomendador de Cursos por Brecha de Habilidades (GAP)

Sistema que, dada una **universidad + carrera**, compara lo que enseña la malla
con lo que pide el mercado laboral, encuentra la **brecha (GAP)** y recomienda
cursos online para cerrarla.

## Flujo del sistema
```
1. Alumno elige universidad + carrera
2. Word2Vec predice qué skills enseña cada curso (sobre el NOMBRE del curso)
3. El alumno marca, por curso, cuáles de esas skills YA sabe
4. Cobertura = skills que el alumno marcó como dominadas
5. GAP = skills demandadas por el mercado NO cubiertas
6. Se recomiendan N cursos online que cierran el GAP (TF-IDF + coseno + MMR)
```

## Técnicas usadas
- **Word2Vec** → predicción de skills por curso + medición de cobertura semántica
- **TF-IDF + similitud coseno** → relevancia curso ↔ GAP
- **MMR** → diversidad del paquete recomendado

## Estructura de carpetas esperada
```
mi_proyecto/
├── recomendador_gap.ipynb
├── datos/
│   ├── tokenizado_cursos.csv
│   ├── df_ofertas_laborales.csv
│   ├── palabras_3.csv
│   ├── coursera_cursos_v5.csv
│   └── udemy_cursos_playwright_3.csv
└── modelos/                  ← se crea sola al entrenar la 1ª vez
```


## 0 · Instalación de librerías

Ejecuta solo si no las tienes.

In [ ]:
# !pip install gensim scikit-learn pandas numpy scipy

## 1 · Imports

In [ ]:
import os
import re
import ast
import pickle
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("modelos", exist_ok=True)
print("Imports OK")


## 2 · Funciones de limpieza de texto

Definimos utilidades y **tres conjuntos de stopwords**:
- `STOP_CURSO`: relleno académico en nombres de cursos
- `STOP_OFERTAS`: relleno de avisos laborales
- `STOP_CURSOS_ONLINE`: verbos genéricos, ruido y duplicados en `palabras_3.csv`

> `df_ofertas_laborales.csv` ya viene limpio. Las stopwords de ofertas se aplican
> igualmente por si se necesita re-limpiar (flag `RELIMPIAR`).


In [ ]:
def quitar_acentos(texto):
    """'cálculo' → 'calculo'."""
    nfkd = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def parse_lista(valor):
    """Convierte un string "['a','b']" en lista de Python."""
    if isinstance(valor, list):
        return valor
    if not isinstance(valor, str) or not valor.strip():
        return []
    try:
        out = ast.literal_eval(valor)
        return out if isinstance(out, list) else [str(out)]
    except (ValueError, SyntaxError):
        return re.findall(r"[a-zA-Z0-9_]+", valor)

def limpiar_tokens(lista, stop=None, min_len=2, dedup=False):
    """Filtra tokens cortos y stopwords. min_len=2 conserva siglas
    técnicas como 'bi', 'js', 'qa'. Con dedup=True elimina repeticiones
    dentro del mismo documento preservando el orden."""
    stop = stop or set()
    resultado = [t for t in lista if len(t) >= min_len and t not in stop]
    if dedup:
        seen = set()
        resultado_dedup = []
        for t in resultado:
            if t not in seen:
                seen.add(t)
                resultado_dedup.append(t)
        return resultado_dedup
    return resultado


# ── Stopwords de NOMBRES DE CURSOS (malla universitaria) ──────────────
# No incluimos 'calculo','fisica','algebra' etc. porque son materias reales
# que Word2Vec puede relacionar con otros conceptos técnicos.
STOP_CURSO = {
    "fundamentos", "introduccion", "electivo", "electivos", "taller",
    "seminario", "laboratorio", "trabajo", "practica", "practicas",
    "proyecto", "proyectos", "aplicaciones", "ingles", "comunicacion",
    "humanidades", "general", "basico", "nivel", "parte", "modulo",
    "area", "aspectos", "ingenieria", "ciencia", "ciencias",
}

# ── Stopwords de OFERTAS LABORALES (para re-limpieza opcional) ─────────
STOP_OFERTAS = {
    "intermedio", "avanzado", "basico", "nivel", "conocimiento",
    "conocimientos", "experiencia", "manejo", "herramienta", "herramientas",
    "requisito", "requisitos", "deseable", "indispensable", "capacidad",
    "habilidad", "habilidades", "competencia", "competencias", "minimo",
    "anos", "ano", "roles", "posiciones", "perfil", "puesto", "soporte",
    "fuerte", "enfoque", "manera", "presencial", "remoto", "horas",
    "realizando", "estudios", "encontramos", "orientado", "generacion",
    "idealmente", "evaluara", "proceso", "asistente", "analista",
    "formar", "estudiante", "sector",
}

# ── Stopwords de CURSOS ONLINE (palabras_3.csv) ────────────────────────
# Verbos genéricos del lenguaje de marketing de cursos, palabras de relleno
# y ruido detectado en el análisis de frecuencias del CSV.
STOP_CURSOS_ONLINE = {
    # Verbos genéricos de marketing de cursos
    "aprender", "aprenderas", "aprendera", "crear", "poder", "tener",
    "utilizar", "usar", "permitir", "realizar", "desarrollar", "disenar",
    "trabajar", "comenzar", "empezar", "conocer", "entender", "comprender",
    "dominar", "gestionar", "manejar", "optimizar", "mejorar", "generar",
    "obtener", "construir", "implementar", "analizar", "aplicar", "llevar",
    "dar", "ver", "saber", "querer", "podras", "incluir", "pasar", "buscar",
    "ensenar", "ayudar", "tomar", "disenado", "finalizar",
    # Sustantivos de relleno del copy de cursos
    "curso", "clase", "leccion", "modulo", "seccion", "video", "material",
    "recurso", "ejercicio", "tarea", "ejemplo", "tema", "parte", "paso",
    "fundamento", "concepto", "nivel", "basico", "avanzado", "practico",
    "completo", "introductorio", "introductoria", "introduccion",
    "programa", "especializado", "formacion", "aprendizaje",
    "conocimiento", "habilidad", "contenido", "objetivo",
    # Adjetivos/adverbios vacíos
    "facil", "rapido", "sencillo", "claro", "efectivo", "diferente",
    "alguno", "mismo", "propio", "nuevo", "importante", "necesario",
    "principal", "dinamico", "largo", "previo", "clave",
    # Palabras temporales / cuantificadoras
    "hora", "dia", "vez", "tiempo", "primero", "segundo", "final",
    "cero", "mundo", "vida", "forma", "manera",
    # Ruido de idioma (inglés mezclado)
    "and", "the", "for", "with", "from", "you", "your",
    # Palabras de audiencia (no skills)
    "estudiante", "alumno", "persona", "profesional", "experto",
    "usuario", "cualquiera", "cliente",
    # Otros sin valor técnico
    "online", "acceso", "ingles", "traves", "ademas", "mediante",
    "dentro", "creer",
}

print(f"STOP_CURSO:          {len(STOP_CURSO)} palabras")
print(f"STOP_OFERTAS:        {len(STOP_OFERTAS)} palabras")
print(f"STOP_CURSOS_ONLINE:  {len(STOP_CURSOS_ONLINE)} palabras")


## 3 · Carga de datos

### 3.1 · Mallas universitarias (`tokenizado_cursos.csv`)

In [ ]:
mallas = pd.read_csv("datos/tokenizado_cursos.csv")
mallas = mallas.rename(columns={"Unnamed: 0": "id"})

mallas["tokens_nombre"] = mallas["tokens"].apply(
    lambda v: limpiar_tokens(parse_lista(v), stop=STOP_CURSO)
)

n_con = (mallas["tokens_nombre"].map(len) > 0).sum()
n_sin = (mallas["tokens_nombre"].map(len) == 0).sum()
print(f"Mallas: {len(mallas)} filas | "
      f"{mallas['universidad'].nunique()} universidades | "
      f"{mallas['carrera'].nunique()} carreras")
print(f"Cursos con tokens técnicos: {n_con}")
print(f"Cursos sin tokens (electivos/inglés/genéricos): {n_sin}")
print()
print("Ejemplos de tokenización:")
for _, r in mallas[mallas["tokens_nombre"].map(len) > 0].head(6).iterrows():
    print(f"  '{r['curso']}' → {r['tokens_nombre']}")


### 3.2 · Ofertas laborales (`df_ofertas_laborales.csv`)

Este CSV **ya viene limpio**. Solo se parsea `habilidades` y se aplican
stopwords como segunda pasada. Cambiar `RELIMPIAR = True` para forzar
re-limpieza completa.


In [ ]:
RELIMPIAR = False   # True para forzar re-limpieza completa

ofertas = pd.read_csv("datos/df_ofertas_laborales.csv")
ofertas["hab_tokens"] = ofertas["habilidades"].apply(parse_lista)

if RELIMPIAR:
    ofertas["hab_tokens"] = ofertas["hab_tokens"].apply(
        lambda l: limpiar_tokens(l, stop=STOP_OFERTAS)
    )
else:
    # Segunda pasada ligera: solo eliminar tokens que entraron como ruido
    ofertas["hab_tokens"] = ofertas["hab_tokens"].apply(
        lambda l: limpiar_tokens(l, stop=STOP_OFERTAS)
    )

ofertas = ofertas[ofertas["hab_tokens"].map(len) > 0].reset_index(drop=True)

print(f"Ofertas válidas: {len(ofertas)}")
print(f"Tokens promedio por oferta: {ofertas['hab_tokens'].map(len).mean():.1f}")
print(f"Niveles: {ofertas['Nivel'].value_counts().to_dict()}")
print()
print("Ejemplo de habilidades:")
for lst in ofertas["hab_tokens"].head(3):
    print(" ", lst[:12])


### 3.3 · Cursos online + metadata (precio, url)

**Limpieza de `palabras_3.csv`:**
- Se eliminan duplicados dentro de cada documento (mismo token repetido por el scraping del texto completo).
- Se aplica `STOP_CURSOS_ONLINE` para quitar verbos genéricos y ruido de marketing.
- Se filtran cursos de idiomas (ensucian recomendaciones técnicas).


In [ ]:
def parsear_precio(valor):
    """'Precio actual62,90 S/' → 'S/ 62.90'; vacío → 'No especificado'."""
    if not isinstance(valor, str) or not valor.strip():
        return "No especificado"
    if "gratis" in valor.lower():
        return "Gratis"
    m = re.search(r"([0-9.]+,[0-9]+|[0-9]+[.,]?[0-9]*)",
                  valor.replace("Precio actual", ""))
    if m:
        try:
            return f"S/ {float(m.group(1).replace('.','').replace(',','.')):.2f}"
        except ValueError:
            pass
    return valor.strip()

# ── Cargar palabras_3.csv (separador '|') ────────────────────────────
online = pd.read_csv("datos/palabras_3.csv", sep="|")
online = online.rename(columns={
    "texto_consolidado": "descripcion",
    "fuente": "portal"
})

# Limpiar tokens: quitar duplicados + stopwords de verbos/ruido
online["tokens"] = online["tokens_limpios"].apply(
    lambda v: limpiar_tokens(
        parse_lista(v),
        stop=STOP_CURSOS_ONLINE,
        dedup=True          # elimina repeticiones dentro del mismo doc
    )
)
online["doc"] = online["tokens"].apply(lambda t: " ".join(t))

# ── Metadata Coursera + Udemy (precio, url) ──────────────────────────
cou  = pd.read_csv("datos/coursera_cursos_v5.csv", sep=";")[["titulo", "precio", "url"]]
ude  = pd.read_csv("datos/udemy_cursos_playwright_3.csv", sep=";")[["titulo", "precio", "url"]]
meta = pd.concat([cou, ude], ignore_index=True).drop_duplicates("titulo", keep="first")
meta["precio"] = meta["precio"].apply(parsear_precio)

online = online.merge(meta, on="titulo", how="left")
online["precio"] = online["precio"].fillna("No especificado")
online["url"]    = online["url"].fillna("")
online = online[online["tokens"].map(len) > 0].reset_index(drop=True)

# ── Filtrar cursos de IDIOMAS ────────────────────────────────────────
PATRON_IDIOMAS = re.compile(
    r"\b(ingl[eé]s|english|franc[eé]s|french|alem[aá]n|german|"
    r"italiano|portugu[eé]s|mandar[ií]n|chino|japon[eé]s|"
    r"idioma|toefl|ielts|gram[aá]tica)\b", re.IGNORECASE
)
n_antes = len(online)
online = online[~online["titulo"].astype(str).apply(
    lambda t: bool(PATRON_IDIOMAS.search(t)))].reset_index(drop=True)

print(f"Cursos de idiomas filtrados: {n_antes - len(online)}")
print(f"Cursos online: {len(online)}")
print(f"Portales: {online['portal'].value_counts().to_dict()}")
print(f"Con precio: {(online['precio'] != 'No especificado').sum()} | "
      f"Con url: {(online['url'] != '').sum()}")
print()
print("Ejemplo de tokens limpios (sin duplicados ni verbos genéricos):")
for _, r in online.head(3).iterrows():
    print(f"  '{r['titulo']}' → {r['tokens'][:10]}")


## 4 · Word2Vec (entrena la 1ª vez, carga después)

Entrenamos **un solo modelo** sobre todo el corpus (cursos online + ofertas +
nombres de cursos universitarios). Así las palabras de los tres mundos quedan
en el mismo espacio vectorial.

Para reentrenar: **borra `modelos/word2vec.model`** y vuelve a ejecutar.


In [ ]:
def entrenar_word2vec(mallas, ofertas, online,
                      vector_size=100, window=5, min_count=2,
                      sg=1, epochs=20, seed=42):
    """Skip-Gram (sg=1) sobre todo el corpus disponible."""
    corpus  = list(online["tokens"])
    corpus += list(ofertas["hab_tokens"])
    corpus += list(mallas["tokens_nombre"])
    corpus  = [d for d in corpus if d]
    return Word2Vec(sentences=corpus, vector_size=vector_size, window=window,
                    min_count=min_count, sg=sg, workers=4,
                    epochs=epochs, seed=seed)

RUTA_W2V = "modelos/word2vec.model"

if os.path.exists(RUTA_W2V):
    modelo = Word2Vec.load(RUTA_W2V)
    print(f"✓ Modelo cargado desde {RUTA_W2V}")
else:
    print("Entrenando Word2Vec (1ª vez, ~60-90 s)...")
    modelo = entrenar_word2vec(mallas, ofertas, online)
    modelo.save(RUTA_W2V)
    print(f"✓ Modelo entrenado y guardado en {RUTA_W2V}")

print(f"Vocabulario: {len(modelo.wv)} palabras")


In [ ]:
def vec(tokens, modelo):
    """Vector promedio de un conjunto de tokens."""
    vs = [modelo.wv[t] for t in tokens if t in modelo.wv]
    return np.mean(vs, axis=0) if vs else None

def cos(a, b):
    """Similitud coseno entre dos vectores (0 si alguno es None)."""
    if a is None or b is None:
        return 0.0
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0

# Prueba rápida de calidad del modelo
for palabra in ["sql", "seguridad", "python"]:
    if palabra in modelo.wv:
        vecinas = [w for w, _ in modelo.wv.most_similar(palabra, topn=5)]
        print(f"'{palabra}' ~ {vecinas}")


## 5 · Predicción de skills por curso ⭐

Para cada curso universitario, Word2Vec predice qué habilidades laborales podría
enseñar. Solo aparecen **skills reales del mercado** (presentes en ≥ MIN_OFERTAS
ofertas y en el vocabulario de Word2Vec).

### Filtro técnico simplificado
En lugar de KMeans, usamos **similitud coseno directa** entre el vector del curso
y un vector de "perfil tecnológico" (promedio de palabras semilla tech). Es más
rápido, más predecible y evita que el agrupamiento varíe con el random seed.


In [ ]:
# ── Vocabulario de habilidades REALES del mercado ────────────────────
MIN_OFERTAS = 5

freq_hab = Counter()
for toks in ofertas["hab_tokens"]:
    freq_hab.update(set(toks))

habilidades_validas = [h for h, n in freq_hab.items()
                       if n >= MIN_OFERTAS and h in modelo.wv]
print(f"Habilidades candidatas (freq>={MIN_OFERTAS} y en Word2Vec): {len(habilidades_validas)}")
print("Ejemplos:", habilidades_validas[:20])


In [ ]:
# ── Filtro 1: exclusión explícita de cursos NO técnicos ───────────────
TOKENS_NO_TECH = {
    # Matemáticas puras / ciencias básicas
    "calculo", "algebra", "fisica", "quimica", "matematica", "matematicas",
    "geometria", "trigonometria", "vectores", "mecanica", "termodinamica",
    # Humanidades / ciencias sociales
    "economia", "contabilidad", "administracion", "finanzas", "contable",
    "geografia", "historia", "filosofia", "sociologia", "politica", "derecho",
    "etica", "religion", "teologia", "psicologia", "antropologia",
    # Comunicación / idiomas
    "redaccion", "escritura", "oratoria", "lenguaje", "literatura",
    "periodismo", "linguistica",
    # Marketing / negocios generales
    "marketing", "publicidad", "ventas", "emprendimiento", "liderazgo",
    "negociacion", "logistica", "tesis",
    # Medio ambiente
    "ambiental", "sostenibilidad", "ecologia", "responsabilidad",
    # Salud
    "salud", "medicina", "enfermeria", "nutricion",
    # Otros sin contenido técnico
    "nivelacion", "deportes", "arte", "musica", "teatro",
}

mallas["es_no_tech"] = mallas["tokens_nombre"].apply(
    lambda t: bool(set(t) & TOKENS_NO_TECH)
)
print(f"Cursos técnicos (pasan Filtro 1): {(~mallas['es_no_tech']).sum()}")
print(f"Cursos NO técnicos (excluidos):   {mallas['es_no_tech'].sum()}")


In [ ]:
# ── Filtro 2 (simplificado): similitud coseno con perfil tech ──────────
# Umbral directo sin KMeans: un curso es técnico si su vector promedio
# supera UMBRAL_SIM_TECH de similitud con el centroide de palabras tech.
# Captura casos ambiguos que el Filtro 1 no eliminó.

SEMILLAS_TECH = [
    "python", "sql", "datos", "algoritmos", "software",
    "redes", "seguridad", "cloud", "programacion", "sistemas",
    "machine_learning", "base_datos", "desarrollo", "automatizacion",
]
UMBRAL_SIM_TECH = 0.30   # ajustar si filtra de más o de menos

semillas_en_vocab = [s for s in SEMILLAS_TECH if s in modelo.wv]
v_tech = np.mean([modelo.wv[s] for s in semillas_en_vocab], axis=0)
print(f"Semillas tech en vocabulario: {semillas_en_vocab}")

# Calcular similitud de cada curso (que pasó Filtro 1) con el perfil tech
def sim_con_tech(tokens_nombre):
    v = vec(tokens_nombre, modelo)
    return cos(v, v_tech) if v is not None else 0.0

candidatos_f1 = mallas[~mallas["es_no_tech"] & (mallas["tokens_nombre"].map(len) > 0)].copy()
candidatos_f1["sim_tech"] = candidatos_f1["tokens_nombre"].apply(sim_con_tech)

cursos_tech_f2 = set(candidatos_f1[candidatos_f1["sim_tech"] >= UMBRAL_SIM_TECH]["curso"])
mallas["es_tech"] = mallas["curso"].isin(cursos_tech_f2)

print(f"\nCursos técnicos finales (ambos filtros): {mallas['es_tech'].sum()}")
print(f"Cursos descartados en total:              {(~mallas['es_tech']).sum()}")
print()
print("Muestra de cursos en el límite del umbral:")
muestra_limite = (candidatos_f1
                  .sort_values("sim_tech")
                  .query("sim_tech >= @UMBRAL_SIM_TECH - 0.05 and sim_tech <= @UMBRAL_SIM_TECH + 0.05"))
for _, r in muestra_limite.head(6).iterrows():
    print(f"  sim={r['sim_tech']:.3f} | {r['curso']} → {r['tokens_nombre']}")


In [ ]:
def predecir_skills(tokens_curso, modelo, habilidades, top_k=6, umbral=0.30):
    """Top-k habilidades laborales más cercanas al nombre del curso.
    umbral: descarta skills con baja similitud semántica."""
    v_curso = vec(tokens_curso, modelo)
    if v_curso is None:
        return []
    scores = [(h, cos(v_curso, modelo.wv[h])) for h in habilidades]
    scores = [(h, s) for h, s in scores if s >= umbral]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [h for h, _ in scores[:top_k]]

# Precalcular para toda la malla (una sola vez)
mallas["skills_predichos"] = mallas["tokens_nombre"].apply(
    lambda t: predecir_skills(t, modelo, habilidades_validas)
)

print("Ejemplos de predicción de skills:")
muestra = mallas[mallas["skills_predichos"].map(len) > 0].sample(8, random_state=1)
for _, r in muestra.iterrows():
    print(f"  '{r['curso']}'")
    print(f"     → {r['skills_predichos']}")


## 6 · Pipeline de recomendación

### 6.1 · Filtrar ofertas relevantes para la carrera


In [ ]:
def filtrar_ofertas_por_carrera(ofertas, carrera, modelo, top_pct=0.5):
    """Conserva las ofertas más afines a la carrera (similitud coseno
    entre el nombre de la carrera y las habilidades de cada oferta)."""
    carrera_tokens = limpiar_tokens(
        quitar_acentos(carrera).lower().split(), stop=set()
    )
    v_carrera = vec(carrera_tokens, modelo)
    sims = [cos(vec(toks, modelo), v_carrera) for toks in ofertas["hab_tokens"]]
    ofertas = ofertas.copy()
    ofertas["sim_carrera"] = sims
    umbral = np.quantile(sims, 1 - top_pct)
    return ofertas[ofertas["sim_carrera"] >= umbral].reset_index(drop=True)

def construir_demanda(ofertas_rel):
    """Frecuencia de cada habilidad en las ofertas relevantes."""
    cnt = Counter()
    for toks in ofertas_rel["hab_tokens"]:
        cnt.update(set(toks))
    return dict(cnt)


### 6.2 · Calcular el GAP

In [ ]:
def calcular_gap(demanda, tokens_cubiertos, modelo, umbral=0.65):
    """Una habilidad demandada es GAP si su mejor similitud con lo cubierto
    queda bajo el umbral. El peso prioriza lo muy pedido y poco cubierto."""
    vecs_cub = {t: modelo.wv[t] for t in tokens_cubiertos if t in modelo.wv}
    filas = []
    for skill, freq in demanda.items():
        if skill in modelo.wv and vecs_cub:
            cobertura = max(cos(modelo.wv[skill], v) for v in vecs_cub.values())
        else:
            cobertura = 0.0
        filas.append({
            "habilidad":  skill,
            "demanda":    freq,
            "cobertura":  round(cobertura, 3),
            "es_gap":     cobertura < umbral,
            "peso_gap":   round(freq * (1 - cobertura), 3),
        })
    return (pd.DataFrame(filas)
            .sort_values("peso_gap", ascending=False)
            .reset_index(drop=True))


### 6.3 · Índice TF-IDF (entrena la 1ª vez, carga después)

Para reajustar: **borra `modelos/tfidf.pkl` y `modelos/tfidf_X.npz`**.


In [ ]:
RUTA_TFIDF   = "modelos/tfidf.pkl"
RUTA_TFIDF_X = "modelos/tfidf_X.npz"

if os.path.exists(RUTA_TFIDF) and os.path.exists(RUTA_TFIDF_X):
    with open(RUTA_TFIDF, "rb") as f:
        tfidf = pickle.load(f)
    X = scipy.sparse.load_npz(RUTA_TFIDF_X)
    print("✓ TF-IDF cargado desde disco")
else:
    print("Ajustando TF-IDF...")
    tfidf = TfidfVectorizer(token_pattern=r"[^ ]+", min_df=2)
    X = tfidf.fit_transform(online["doc"])
    with open(RUTA_TFIDF, "wb") as f:
        pickle.dump(tfidf, f)
    scipy.sparse.save_npz(RUTA_TFIDF_X, X)
    print("✓ TF-IDF ajustado y guardado")

print(f"Matriz TF-IDF: {X.shape[0]} cursos × {X.shape[1]} términos")


### 6.4 · Recomendar: TF-IDF + coseno + MMR

In [ ]:
def recomendar_cursos(online, gap_df, modelo, tfidf, X,
                      n_cursos=5, top_k_gap=40, n_candidatos=120,
                      alpha=1.0):
    """
    Pipeline de recomendación:
      (a) TF-IDF + coseno → ranking de relevancia curso ↔ GAP
      (b) MMR              → selección diversa que maximiza cobertura
                             y minimiza redundancia

    alpha: peso de la relevancia TF-IDF en el score inicial (0-1).
    """
    gap = gap_df[gap_df["es_gap"]].sort_values("peso_gap", ascending=False)
    if gap.empty:
        gap = gap_df.copy()
    gap = gap.head(top_k_gap)

    # (a) TF-IDF + coseno: relevancia curso ↔ GAP
    consulta   = " ".join(gap["habilidad"])
    q          = tfidf.transform([consulta])
    relevancia = cosine_similarity(q, X).ravel()

    M = min(n_candidatos, int((relevancia > 0).sum()))
    if M == 0:
        print("⚠ Ningún curso coincide con el GAP.")
        return pd.DataFrame()

    idx  = np.argsort(relevancia)[::-1][:M]
    cand = online.iloc[idx].copy().reset_index(drop=True)
    cand["relevancia"] = relevancia[idx]

    def norm(s):
        r = s.max() - s.min()
        return (s - s.min()) / r if r > 0 else s * 0 + 1.0

    cand["score"] = norm(cand["relevancia"])
    cand = cand.sort_values("score", ascending=False).reset_index(drop=True)

    # (b) MMR: cobertura del GAP + anti-redundancia
    peso_skill = dict(zip(gap["habilidad"], gap["peso_gap"]))
    total      = sum(peso_skill.values()) or 1.0
    cand_sets  = [set(t) & set(peso_skill) for t in cand["tokens"]]
    Xc2        = tfidf.transform(cand["doc"])

    elegidos, cubiertas = [], set()
    while len(elegidos) < min(n_cursos, len(cand)):
        mejor_i, mejor_val = None, -1e9
        for i in range(len(cand)):
            if i in elegidos:
                continue
            nuevas    = cand_sets[i] - cubiertas
            cobertura = sum(peso_skill[s] for s in nuevas) / total
            redund    = (cosine_similarity(Xc2[i], Xc2[elegidos]).max()
                         if elegidos else 0.0)
            val = 0.35 * cand.loc[i, "score"] + 0.45 * cobertura - 0.40 * redund
            if val > mejor_val:
                mejor_val, mejor_i = val, i
        elegidos.append(mejor_i)
        cubiertas |= cand_sets[mejor_i]

    cand = cand.iloc[elegidos].reset_index(drop=True)

    # Explicabilidad: qué skills del GAP cubre cada curso
    cand["gap_cubierto"] = cand["tokens"].apply(
        lambda t: ", ".join(sorted(set(t) & set(peso_skill))[:8]) or "—")

    cols = ["titulo", "descripcion", "portal", "precio", "url",
            "relevancia", "score", "gap_cubierto"]
    out = cand[cols].copy()
    out["descripcion"] = out["descripcion"].astype(str).str[:200] + "..."
    out["relevancia"]  = out["relevancia"].round(3)
    out["score"]       = out["score"].round(3)
    return out

print("Función recomendar_cursos definida ✓")


## 7 · Función principal `recomendar()`

In [ ]:
def recomendar(universidad, carrera, skills_que_se=None, niveles=None,
               n_cursos=5, top_pct=0.5, umbral_gap=0.65, verbose=True):
    """
    Parámetros
    ----------
    universidad   : nombre exacto de la universidad (ver combos disponibles)
    carrera       : nombre exacto de la carrera
    skills_que_se : lista/set de skills que el alumno ya domina
    niveles       : lista de niveles de oferta a considerar (None = todos)
    n_cursos      : número de cursos a recomendar
    top_pct       : fracción de ofertas más afines a la carrera (0.0-1.0)
    umbral_gap    : umbral de cobertura semántica para declarar una skill como GAP
    verbose       : imprime resumen en pantalla
    """
    skills_que_se = set(skills_que_se or [])

    sub = mallas[(mallas["universidad"] == universidad) &
                 (mallas["carrera"] == carrera)]
    if sub.empty:
        raise ValueError(f"No hay malla para ({universidad} | {carrera})")

    ofr = ofertas.copy()
    if niveles:
        ofr = ofr[ofr["Nivel"].isin(niveles)].reset_index(drop=True)
    ofr_rel = filtrar_ofertas_por_carrera(ofr, carrera, modelo, top_pct)
    demanda = construir_demanda(ofr_rel)

    gap_df = calcular_gap(demanda, sorted(skills_que_se), modelo, umbral_gap)
    recs   = recomendar_cursos(online, gap_df, modelo, tfidf, X, n_cursos=n_cursos)

    info = {
        "skills_dominadas":       len(skills_que_se),
        "ofertas_relevantes":     len(ofr_rel),
        "habilidades_demandadas": len(demanda),
        "gap":                    int(gap_df["es_gap"].sum()),
    }
    if verbose:
        print(f"\n=== {universidad} | {carrera} ===")
        for k, v in info.items():
            print(f"  {k}: {v}")
        print("\nTop habilidades del GAP:")
        print(gap_df[gap_df["es_gap"]].head(10)
              [["habilidad", "demanda", "cobertura", "peso_gap"]]
              .to_string(index=False))
        print(f"\nTop {n_cursos} cursos recomendados:")
        for _, r in recs.iterrows():
            print(f"  • [{r['portal']}] {r['titulo']}  ({r['precio']})")
            print(f"      cubre: {r['gap_cubierto']}")
    return recs, gap_df, info

print("Función recomendar() definida ✓")


## 8 · Análisis de huecos temáticos de la malla

In [ ]:
def detectar_huecos(universidad, carrera, modelo, top_pct=0.5,
                    umbral_cobertura=0.40, top_n=10):
    """
    Compara las skills demandadas por el mercado contra TODO lo que enseña
    la malla (solo cursos técnicos). Las skills sin cobertura son 'huecos
    temáticos': áreas que la universidad no enseña en absoluto.

    umbral_cobertura es más bajo que calcular_gap (0.40 vs 0.65) porque
    aquí queremos huecos REALES de la malla completa, no del alumno.
    """
    sub = mallas[
        (mallas["universidad"] == universidad) &
        (mallas["carrera"] == carrera) &
        (mallas["es_tech"])
    ]
    if sub.empty:
        print("No hay cursos técnicos en esta malla.")
        return pd.DataFrame(), pd.DataFrame()

    tokens_malla = sorted({t for toks in sub["tokens_nombre"] for t in toks})
    ofr_rel  = filtrar_ofertas_por_carrera(ofertas, carrera, modelo, top_pct)
    demanda  = construir_demanda(ofr_rel)

    vecs_malla = {t: modelo.wv[t] for t in tokens_malla if t in modelo.wv}
    huecos = []
    for skill, freq in demanda.items():
        if skill not in modelo.wv or not vecs_malla:
            cobertura = 0.0
        else:
            cobertura = max(cos(modelo.wv[skill], v) for v in vecs_malla.values())
        if cobertura < umbral_cobertura:
            huecos.append({
                "skill":            skill,
                "demanda":          freq,
                "cobertura_malla":  round(cobertura, 3),
                "peso_hueco":       round(freq * (1 - cobertura), 3),
            })

    huecos_df = (pd.DataFrame(huecos)
                 .sort_values("peso_hueco", ascending=False)
                 .reset_index(drop=True))

    print(f"Huecos temáticos para {carrera} en {universidad}:")
    print(f"  Skills demandadas:                {len(demanda)}")
    print(f"  Huecos (no cubiertas por malla):  {len(huecos_df)}")
    print(f"\nTop {top_n} huecos:")
    print(huecos_df.head(top_n)[["skill", "demanda", "cobertura_malla", "peso_hueco"]]
          .to_string(index=False))

    # Cursos online que cierran los huecos
    gap_huecos = huecos_df.rename(columns={"skill": "habilidad", "peso_hueco": "peso_gap"})
    gap_huecos["es_gap"]     = True
    gap_huecos["cobertura"]  = gap_huecos["cobertura_malla"]

    recs_huecos = recomendar_cursos(online, gap_huecos, modelo, tfidf, X, n_cursos=5)
    print(f"\nCursos recomendados para cubrir los huecos:")
    for _, r in recs_huecos.iterrows():
        print(f"  • [{r['portal']}] {r['titulo']}  ({r['precio']})")
        print(f"      cubre: {r['gap_cubierto']}")

    return huecos_df, recs_huecos

print("Función detectar_huecos() definida ✓")


## 9 · Prueba del pipeline

In [ ]:
# Combos disponibles
combos = mallas.groupby(["universidad", "carrera"]).size().reset_index(name="n_cursos")
print("Combos disponibles:")
print(combos.to_string(index=False))


In [ ]:
MI_UNIVERSIDAD = "Universidad del Pacífico (UP)"
MI_CARRERA     = "Ingeniería de la Información"

# Ver los cursos de esa carrera con sus skills predichos
sub = mallas[(mallas["universidad"] == MI_UNIVERSIDAD) &
             (mallas["carrera"] == MI_CARRERA)]
sub = sub[sub["skills_predichos"].map(len) > 0].sort_values("ciclo")

print(f"Cursos de {MI_CARRERA} (con skills predichos):\n")
for _, r in sub.iterrows():
    print(f"  Ciclo {r['ciclo']} | {r['curso']}")
    print(f"     skills: {r['skills_predichos']}")


In [ ]:
# Simular: el alumno domina las skills de la primera mitad de la carrera
skills_dominadas = set()
for _, r in sub.head(len(sub) // 2).iterrows():
    skills_dominadas.update(r["skills_predichos"])

print(f"Skills que el alumno marcó como dominadas ({len(skills_dominadas)}):")
print(sorted(skills_dominadas))

recs, gap_df, info = recomendar(
    universidad   = MI_UNIVERSIDAD,
    carrera       = MI_CARRERA,
    skills_que_se = skills_dominadas,
    niveles       = None,
    n_cursos      = 5,
)


In [ ]:
# Tabla final de recomendaciones
recs[["titulo", "portal", "precio", "score", "gap_cubierto"]]


## 10 · Verificación de reactividad

A menos skills dominadas → más GAP → más cursos recomendados (distintos).


In [ ]:
todas_skills = sorted({s for toks in sub["skills_predichos"] for s in toks})

resultados = []
for frac in [0.0, 0.25, 0.5, 1.0]:
    n = int(len(todas_skills) * frac)
    marcadas = set(todas_skills[:n])
    _, g, inf = recomendar(MI_UNIVERSIDAD, MI_CARRERA,
                           skills_que_se=marcadas, verbose=False)
    resultados.append({"skills_marcadas": n, "gap": inf["gap"]})

print(pd.DataFrame(resultados).to_string(index=False))
print("\n✓ A menos skills marcadas → más GAP.")
